In [ ]:
import pandas as pd
import numpy as np

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# 🔹 Load dataset
df = pd.read_csv("/content/IMDB Dataset.csv")

# 🔹 Convert labels to 0/1
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# 🔹 Split data
X_train, X_test, y_train, y_test = train_test_split(
    df['review'], df['sentiment'], test_size=0.2, random_state=42
)

# 🔹 Tokenization
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# 🔹 Padding
max_len = 200
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

# 🔹 Model
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=64))
model.add(LSTM(64))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# 🔹 Train
model.fit(X_train_pad, y_train, epochs=5, batch_size=128, validation_split=0.2)

# 🔹 Evaluate
loss, acc = model.evaluate(X_test_pad, y_test)
print("Test Accuracy:", acc)

def predict_sentiment(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)

    prob = model.predict(padded)[0][0]

    if prob >= 0.5:
        return f"Positive 😊 ({prob:.2f})"
    else:
        return f"Negative 😞 ({prob:.2f})"

print(predict_sentiment("This movie was fantastic!"))
print(predict_sentiment("I hated this film"))

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 75s 293ms/step - accuracy: 0.8117 - loss: 0.4027 - val_accuracy: 0.8754 - val_loss: 0.2970
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 72s 289ms/step - accuracy: 0.8937 - loss: 0.2650 - val_accuracy: 0.8820 - val_loss: 0.2905
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 73s 290ms/step - accuracy: 0.9104 - loss: 0.2257 - val_accuracy: 0.8719 - val_loss: 0.2998
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 72s 289ms/step - accuracy: 0.9207 - loss: 0.2032 - val_accuracy: 0.8726 - val_loss: 0.3340
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 72s 285ms/step - accuracy: 0.9323 - loss: 0.1760 - val_accuracy: 0.8746 - val_loss: 0.3341
313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - accuracy: 0.8820 - loss: 0.3143
Test Accuracy: 0.8820000290870667
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step
Positive 😊 (0.61)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Negative 😞 (0.22)
